# 🎬 Clip Mining with Qwen3.5-4B + HF Authentication

**Purpose:** Detect viral clip candidates using **Qwen3.5-4B** with Hugging Face authentication

**Model:** `Qwen/Qwen3.5-4B` (4B parameters, fits on T4 with FP16)

**Requirements:**
- FREE Tesla T4 GPU (16GB VRAM)
- Hugging Face Token (for gated model access)

---

## ⚠️ IMPORTANT: Before Running

### 1. Enable GPU Runtime
   - Click `Runtime` → `Change runtime type`
   - Select `GPU` → `T4`
   - Click `Save`

### 2. Get Hugging Face Token
   - Visit: https://huggingface.co/settings/tokens
   - Create new token (Read access is enough)
   - Copy token (starts with `hf_...`)

### 3. Add Token to Colab Secrets
   - Click gear icon ⚙️ (Settings)
   - Go to `Secrets` tab
   - Click `Add secret`
   - Name: `HF_TOKEN`
   - Value: `hf_xxxxxxxxxxxxxxxxxxxxx`
   - Click `OK`

---

## Step 1: Install Dependencies (Latest Transformers + Linear Attention)

In [ ]:
# Install latest Transformers from GitHub (required for Qwen3.5-4B)
print("Installing latest transformers from GitHub...")
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Install linear attention for efficient long-context processing
# This enables Gated Deltanet/Linear attention used in Qwen3.5
print("Installing linear attention support...")
!pip install -q einops flash-attn --no-build-isolation

# Install other dependencies
print("Installing torch and accelerate...")
!pip install -q torch accelerate huggingface_hub

# Verify installations
print("\n" + "="*60)
print("INSTALLED PACKAGES")
print("="*60)
!python -c "import transformers; print(f'transformers: {transformers.__version__}')"
!python -c "import torch; print(f'torch: {torch.__version__}')"
!python -c "import accelerate; print(f'accelerate: {accelerate.__version__}')"
try:
    import flash_attn
    print(f'flash-attn: Installed ✓')
except:
    print(f'flash-attn: Not available (will use standard attention)')
print("="*60)
print("✓ All dependencies installed successfully!")

## Step 2: Hugging Face Authentication

In [ ]:
import os
from huggingface_hub import login, HfApi

print("="*60)
print("HUGGING FACE AUTHENTICATION")
print("="*60)

# Get token from Colab Secrets
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    print("✓ Found HF_TOKEN in Colab Secrets")
except userdata.SecretNotFoundError:
    print("❌ HF_TOKEN not found in Secrets!")
    print("\nPlease add HF_TOKEN to Colab Secrets:")
    print("  1. Click ⚙️ Settings")
    print("  2. Go to Secrets tab")
    print("  3. Add secret: HF_TOKEN")
    print("  4. Paste your token from: https://huggingface.co/settings/tokens")
    raise SystemExit("HF_TOKEN required for Qwen3.5-4B")

# Login to Hugging Face
try:
    login(token=hf_token)
    api = HfApi()
    user = api.whoami(token=hf_token)
    print(f"✓ Logged in as: {user['name']}")
    print(f"  Email: {user.get('email', 'N/A')}")
    
    # Check if user has access to Qwen3.5
    try:
        api.model_info("Qwen/Qwen3.5-4B", token=hf_token)
        print(f"✓ Access confirmed: Qwen/Qwen3.5-4B")
    except Exception as e:
        print(f"⚠️ Model access issue: {e}")
        print("   Make sure you accepted the model license on Hugging Face")
        
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    raise

print("="*60)

## Step 3: Load Qwen3.5-4B Model

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime

# Model configuration
MODEL_NAME = "Qwen/Qwen3.5-4B"  # Using Qwen3.5-4B with HF auth
# MODEL_NAME = "Qwen/Qwen3.5-4B-Base"  # Alternative: base model

VALID_TOPICS = [
    "Charity", "Oppression", "Dua", "Mercy", "Death",
    "Tawakkul", "Sabr", "Afterlife", "Faith", "Prayer",
    "Quran", "Hadith", "Other"
]

# Check GPU
print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ GPU: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB")
    
    # Check if T4
    if "T4" in gpu_name:
        print(f"  ✓ Tesla T4 detected - perfect for Qwen3.5-4B!")
        print(f"  Expected VRAM usage: ~10-12 GB")
        print(f"  Available VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected - Qwen3.5-4B requires GPU!")
    print("Please enable GPU: Runtime → Change runtime type → GPU (T4)")
    raise SystemExit("GPU required")
print("="*60)

# Load model with HF token and optimizations
print(f"\nLoading model: {MODEL_NAME}")
print(f"Using HF authentication: {'✓ Yes' if hf_token else '✗ No'}")
print("Using linear attention optimizations: ✓ Enabled")
print("This may take 2-3 minutes for first download...\n")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=hf_token,
        trust_remote_code=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,  # FP16 for T4 efficiency
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        token=hf_token,  # HF authentication for gated model
        attn_implementation="flash_attention_2"  # Use flash attention if available
    )
    
    print(f"✓ Model loaded successfully!")
    print(f"  Device: {model.device}")
    print(f"  VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
    print(f"  VRAM free: {torch.cuda.get_device_properties(0).total_memory / 1024**3 - torch.cuda.memory_allocated() / 1024**3:.1f} GB")
    
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("\nTroubleshooting:")
    print("  1. Check HF_TOKEN is valid")
    print("  2. Accept model license: https://huggingface.co/Qwen/Qwen3.5-4B")
    print("  3. Make sure you have GPU enabled")
    raise

## Step 4: Clip Detection Functions

In [ ]:
# Keywords that indicate important Islamic content
IMPORTANT_KEYWORDS = [
    "allah", "prophet", "quran", "hadith", "islam", "muslim",
    "faith", "belief", "prayer", "salah", "zakat", "sadaqah",
    "charity", "oppression", "dua", "supplication", "mercy",
    "rahmah", "forgiveness", "paradise", "jannah", "hellfire",
    "death", "grave", "afterlife", "akhirah", "sabr", "patience"
]

EMOTIONAL_KEYWORDS = [
    "love", "mercy", "fear", "hope", "beautiful", "amazing",
    "powerful", "heart", "soul", "weep", "cry", "remember",
    "paradise", "hellfire", "forgiveness"
]

def has_important_content(text: str) -> bool:
    text_lower = text.lower()
    return any(kw in text_lower for kw in IMPORTANT_KEYWORDS)

def calculate_clip_score(text: str, topic: str) -> int:
    score = 5
    text_lower = text.lower()
    
    if any(word in text_lower for word in EMOTIONAL_KEYWORDS):
        score += 2
    if any(word in text_lower for word in ['allah', 'prophet', 'quran', 'faith']):
        score += 1
    if any(word in text_lower for word in ['remember', 'learn', 'understand', 'lesson']):
        score += 1
    if topic in ['Charity', 'Oppression', 'Dua', 'Mercy', 'Patience', 'Afterlife']:
        score += 1
    
    return min(score, 10)

def classify_topic_llm(text: str) -> str:
    """Use Qwen3.5-4B to classify topic."""
    prompt = f"""Classify the topic of this Islamic lecture segment.

Topics: {', '.join(VALID_TOPICS[:8])}

Text: {text[:500]}

Return topic only."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.3,
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    topic = response.strip().title()
    
    return topic if topic in VALID_TOPICS else "Other"

def process_chunk(chunk: dict) -> dict:
    text = chunk.get("text", "")
    
    if not has_important_content(text):
        return {
            **chunk,
            "topic": "Other",
            "emotion_score": 3,
            "clip_candidate": False,
            "skipped": True
        }
    
    topic = classify_topic_llm(text)
    emotion_score = calculate_clip_score(text, topic)
    clip_candidate = (topic != "Other" and emotion_score >= 7)
    
    return {
        **chunk,
        "topic": topic,
        "emotion_score": emotion_score,
        "clip_candidate": clip_candidate,
        "skipped": False
    }

## Step 5: Load Transcripts

In [ ]:
print("Loading transcripts...")

# Try different paths
possible_paths = [
    "/content/chunks.json",
    "/content/drive/MyDrive/video_pipeline/chunks.json",
    "chunks.json"
]

chunks = None
used_path = None

for path in possible_paths:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            chunks = data if isinstance(data, list) else data.get('chunks', [])
            used_path = path
            break

if not chunks:
    print("❌ No chunks.json found!")
    print("\nUpload chunks.json to:")
    print("  - /content/chunks.json (direct upload)")
    print("  - /content/drive/MyDrive/video_pipeline/chunks.json (via Drive)")
    raise SystemExit("chunks.json required")
else:
    print(f"✓ Loaded {len(chunks)} chunks from {used_path}")

## Step 6: Process with Qwen3.5-4B GPU

In [ ]:
print("\n" + "="*60)
print("PROCESSING WITH QWEN3.5-4B")
print("="*60)

processed_chunks = []
skipped_count = 0

for i, chunk in enumerate(chunks):
    result = process_chunk(chunk)
    processed_chunks.append(result)
    
    if result.get("skipped"):
        skipped_count += 1
    
    # Progress every 50 chunks
    if (i + 1) % 50 == 0:
        candidates_so_far = sum(1 for c in processed_chunks if c.get("clip_candidate"))
        print(f"  Processed {i+1}/{len(chunks)} ({candidates_so_far} clip candidates)")

# Summary
candidates = [c for c in processed_chunks if c.get("clip_candidate")]
high_scores = [c for c in processed_chunks if c.get("emotion_score", 0) >= 8]

print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Total chunks: {len(chunks)}")
print(f"Skipped (generic): {skipped_count}")
print(f"Processed with Qwen3.5-4B: {len(chunks) - skipped_count}")
print(f"Clip candidates (score ≥7): {len(candidates)}")
print(f"High priority (score ≥8): {len(high_scores)}")
print("="*60)

## Step 7: Merge Adjacent Clips

In [ ]:
def parse_timestamp(timestamp) -> float:
    if not timestamp:
        return 0.0
    parts = str(timestamp).replace(".", ":").split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return 0.0

def merge_adjacent_clips(candidates: list, max_duration: int = 60, min_duration: int = 15) -> list:
    if not candidates:
        return []
    
    sorted_candidates = sorted(
        candidates,
        key=lambda x: parse_timestamp(x.get("timestamp", x.get("timestamp_start", "00:00")))
    )
    
    merged_clips = []
    current_clip = None
    
    for candidate in sorted_candidates:
        ts = candidate.get("timestamp_start") or candidate.get("timestamp")
        te = candidate.get("timestamp_end") or (float(ts) + 8 if ts else "00:00")
        
        start = parse_timestamp(ts)
        end = parse_timestamp(te) if isinstance(te, str) else start + 8
        
        if current_clip is None:
            current_clip = {
                "video_id": candidate.get("video_id", "unknown"),
                "timestamp_start": ts,
                "timestamp_end": te if isinstance(te, str) else f"{int(start//60):02d}:{int(start%60):02d}",
                "start_seconds": start,
                "end_seconds": end,
                "topics": [candidate.get("topic")],
                "emotion_scores": [candidate.get("emotion_score", 0)],
                "chunks": [candidate]
            }
        else:
            time_gap = start - current_clip["end_seconds"]
            
            if time_gap <= 10:
                current_clip["end_seconds"] = end
                current_clip["topics"].append(candidate.get("topic"))
                current_clip["emotion_scores"].append(candidate.get("emotion_score", 0))
                current_clip["chunks"].append(candidate)
            else:
                duration = current_clip["end_seconds"] - current_clip["start_seconds"]
                if min_duration <= duration <= max_duration:
                    topic_counts = {t: current_clip["topics"].count(t) for t in set(current_clip["topics"])}
                    primary_topic = max(topic_counts, key=topic_counts.get)
                    
                    merged_clips.append({
                        "video_id": current_clip["video_id"],
                        "timestamp_start": current_clip["timestamp_start"],
                        "timestamp_end": current_clip["timestamp_end"],
                        "start_seconds": current_clip["start_seconds"],
                        "end_seconds": current_clip["end_seconds"],
                        "duration": duration,
                        "topic": primary_topic,
                        "all_topics": list(set(current_clip["topics"])),
                        "emotion_score": max(current_clip["emotion_scores"]),
                        "chunk_count": len(current_clip["chunks"])
                    })
                
                current_clip = {
                    "video_id": candidate.get("video_id", "unknown"),
                    "timestamp_start": ts,
                    "timestamp_end": te if isinstance(te, str) else f"{int(start//60):02d}:{int(start%60):02d}",
                    "start_seconds": start,
                    "end_seconds": end,
                    "topics": [candidate.get("topic")],
                    "emotion_scores": [candidate.get("emotion_score", 0)],
                    "chunks": [candidate]
                }
    
    if current_clip:
        duration = current_clip["end_seconds"] - current_clip["start_seconds"]
        if min_duration <= duration <= max_duration:
            topic_counts = {t: current_clip["topics"].count(t) for t in set(current_clip["topics"])}
            primary_topic = max(topic_counts, key=topic_counts.get)
            
            merged_clips.append({
                "video_id": current_clip["video_id"],
                "timestamp_start": current_clip["timestamp_start"],
                "timestamp_end": current_clip["timestamp_end"],
                "start_seconds": current_clip["start_seconds"],
                "end_seconds": current_clip["end_seconds"],
                "duration": duration,
                "topic": primary_topic,
                "all_topics": list(set(current_clip["topics"])),
                "emotion_score": max(current_clip["emotion_scores"]),
                "chunk_count": len(current_clip["chunks"])
            })
    
    return merged_clips


# Merge clips
if processed_chunks:
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    merged = merge_adjacent_clips(candidates)
    
    print(f"\n✓ Merged {len(candidates)} candidates into {len(merged)} clips")
    print("\nTop clips (Qwen3.5-4B):")
    for i, clip in enumerate(sorted(merged, key=lambda x: -x['emotion_score'])[:10], 1):
        print(
            f"  {i:2d}. [{clip['topic']:12}] {clip['timestamp_start']}-{clip['timestamp_end']} | "
            f"Score: {clip['emotion_score']} | Duration: {clip['duration']:.0f}s"
        )

## Step 8: Save to Google Drive (FIXED)

In [ ]:
# FIRST: Mount Google Drive
print("="*60)
print("MOUNTING GOOGLE DRIVE")
print("="*60)

from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    print("✓ Google Drive mounted successfully")
except Exception as e:
    print(f"⚠️ Drive mount issue: {e}")
    print("Trying to continue...")

# Ensure output directory exists
drive_output_dir = "/content/drive/MyDrive/video_pipeline"
os.makedirs(drive_output_dir, exist_ok=True)
print(f"✓ Output directory: {drive_output_dir}")
print("="*60)

# Check if we have clips to save
if 'merged' not in locals():
    print("❌ No clips to save! 'merged' variable not found.")
    print("\nPossible issues:")
    print("  1. Step 7 (Merge Clips) was not run")
    print("  2. No clip candidates were found")
    print("  3. Processing failed in earlier steps")
    print("\nPlease run Step 7 first!")
    raise SystemExit("merged variable not found")

if not merged:
    print("❌ No clips to save! 'merged' list is empty.")
    print("\nThis means no clip candidates were found in your transcripts.")
    print("Possible reasons:")
    print("  1. Transcripts don't contain Islamic content keywords")
    print("  2. Emotion scores were all below threshold (<7)")
    print("\nTry with different transcript data.")
    raise SystemExit("merged list is empty")

# Save to Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save JSON
json_file = f"{drive_output_dir}/clip_candidates_qwen35_4b_{timestamp}.json"
try:
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    print(f"\n✓ Saved JSON: {json_file}")
    print(f"  File size: {os.path.getsize(json_file) / 1024:.1f} KB")
except Exception as e:
    print(f"\n❌ Failed to save JSON: {e}")
    print("Check if Drive is properly mounted.")
    raise

# Save CSV
csv_file = f"{drive_output_dir}/clip_candidates_qwen35_4b_{timestamp}.csv"
try:
    with open(csv_file, 'w', encoding='utf-8') as f:
        f.write("video_id,topic,start_time,end_time,duration,emotion_score\n")
        for clip in merged:
            f.write(f"\"{clip['video_id']}\",\"{clip['topic']}\",\"{clip['timestamp_start']}\",\"{clip['timestamp_end']}\",{clip['duration']:.0f},{clip['emotion_score']}\n")
    print(f"✓ Saved CSV: {csv_file}")
except Exception as e:
    print(f"❌ Failed to save CSV: {e}")

# Verify files exist
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
if os.path.exists(json_file):
    print(f"✓ JSON file exists: {json_file}")
else:
    print(f"❌ JSON file NOT created: {json_file}")

if os.path.exists(csv_file):
    print(f"✓ CSV file exists: {csv_file}")
else:
    print(f"❌ CSV file NOT created: {csv_file}")

# Show ffmpeg commands
print("\n" + "="*60)
print("FFMPEG EXTRACTION COMMANDS")
print("="*60)
for i, clip in enumerate(merged[:15], 1):
    topic = clip['topic'].lower().replace(' ', '_')
    print(f"# Clip {i:2d}: {clip['topic']} (Score: {clip['emotion_score']})")
    print(f"ffmpeg -ss {clip['timestamp_start']} -to {clip['timestamp_end']} -i video.mp4 clip_{i:02d}_{topic}.mp4\n")

print("\n" + "="*60)
print("✅ FILES SAVED TO GOOGLE DRIVE")
print("="*60)
print(f"JSON: {json_file}")
print(f"CSV:  {csv_file}")
print(f"\n📁 Check your Google Drive: video_pipeline/ folder")
print(f"\n🎉 Processing complete with Qwen3.5-4B!")

## ✅ Done!

Your clip candidates have been processed with **Qwen3.5-4B** and saved to Google Drive.

### Next Steps:

1. **Check Google Drive:** Open `video_pipeline/clip_candidates_qwen35_4b_*.json`
2. **Download to local machine**
3. **Extract video clips** using ffmpeg commands above
4. **Import to database:**
   ```bash
   python -m clip_pipeline.run_pipeline \
       --transcript clip_candidates_qwen35_4b_TIMESTAMP.json \
       --video-id your_video_id \
       --title "Your Video Title"
   ```